<a href="https://colab.research.google.com/github/myoungjinahn/Machine-Learning-project-team1/blob/main/%EB%9D%BC%EB%B2%A8%EB%A7%81)%EC%98%A8%EB%8F%84%2C%EC%B4%88%EB%AF%B8%EC%84%B8%EB%A8%BC%EC%A7%80_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%A4%80%EB%B9%84_%EA%B3%BC%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ㅇㅁ

1. 파일 이름 설정 및 파일 업로드

In [23]:
import pandas as pd
import os
from google.colab import files

# 1. 파일 이름 설정
pm25_file = "2013~2025년 초미세먼지 가을 데이터 완성본 (9).xlsx"
temp_file = "2013~2025 가을 온도데이터.xlsx"

# 2. 파일 업로드
print("지정된 이름의 두 파일을 업로드해주세요.")
uploaded = files.upload()

지정된 이름의 두 파일을 업로드해주세요.


Saving 2013~2025 가을 온도데이터.xlsx to 2013~2025 가을 온도데이터 (9).xlsx
Saving 2013~2025년 초미세먼지 가을 데이터 완성본.xlsx to 2013~2025년 초미세먼지 가을 데이터 완성본 (9).xlsx


2. 각 데이터 처리 및 데이터 병합 및 라벨링

In [46]:
def process_fix():
    # [A] 초미세먼지 데이터: 구조 기반 전수 추출
    df_pm_raw = pd.read_excel(pm25_file)

    pm_all_list = []
    # 데이터가 시작되는 행부터 끝까지 순회 (보통 0번 컬럼에 지역, 1번 컬럼에 연월)
    for i in range(len(df_pm_raw)):
        row = df_pm_raw.iloc[i]
        ym_val = str(row.iloc[1]) # 두 번째 열에서 '20XX년XX월' 추출

        # '년'과 '월'이 포함된 행만 데이터로 간주
        if '년' in ym_val and '월' in ym_val:
            for day in range(1, 32):
                col_name = f"{day}일"
                if col_name in df_pm_raw.columns:
                    val = row[col_name]
                    if pd.notna(val) and str(val).strip() not in ['-', '']:
                        pm_all_list.append({
                            'Date_Key': f"{ym_val.strip()}_{day}일",
                            'PM25': float(val)
                        })

    df_pm_total = pd.DataFrame(pm_all_list)

    # [B] 온도 데이터 처리
    df_temp_raw = pd.read_excel(temp_file)
    temp_all_list = []
    for _, row in df_temp_raw.iterrows():
        try:
            dt = pd.to_datetime(row.iloc[0])
            # 형식 통일 (예: 2013년09월_1일)
            date_key = f"{dt.year}년{dt.month:02d}월_{dt.day}일"
            temp_all_list.append({'Date_Key': date_key, 'Temp': float(row.iloc[1])})
        except:
            continue
    df_temp_total = pd.DataFrame(temp_all_list)

    # [C] 데이터 병합 및 라벨링
    merged = pd.merge(df_pm_total, df_temp_total, on='Date_Key')

    # 복합 조건 라벨링
    merged['Label'] = merged.apply(
        lambda r: 1 if r['PM25'] <= 35 and 14 <= r['Temp'] <= 22 else 0, axis=1
    )

    return merged

3. 실행 및 저장

In [47]:
try:
    final_df = process_fix()

    if final_df.empty:
        print("⚠️ 데이터가 매칭되지 않았습니다. 파일 형식을 다시 확인해주세요.")
    else:
        print(f"총 {len(final_df)}건의 데이터가 처리되었습니다.")
        print(f"기간: {final_df['Date_Key'].iloc[0]} ~ {final_df['Date_Key'].iloc[-1]}")

        # 연도별 데이터 개수 출력하여 확인
        final_df['Year'] = final_df['Date_Key'].str[:4]
        print("\n[연도별 추출 결과]")
        print(final_df.groupby('Year').size())

        # 파일 저장 및 다운로드
        output_name = 'total_labeled_2013_2025.csv'
        final_df.drop(columns=['Year']).to_csv(output_name, index=False, encoding='utf-8-sig')
        files.download(output_name)

except Exception as e:
    print(f"오류 발생: {e}")

✅ 총 1183건의 데이터가 처리되었습니다.
✅ 기간: 2013년09월_1일 ~ 2025년11월_30일

[연도별 추출 결과]
Year
2013    91
2014    91
2015    91
2016    91
2017    91
2018    91
2019    91
2020    91
2021    91
2022    91
2023    91
2024    91
2025    91
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
import pandas as pd
import io
from google.colab import files

# 1. 분석할 결과 파일(.csv) 업로드
print("분석할 CSV 파일을 업로드해주세요.")
uploaded = files.upload()

# 2. 업로드된 파일 목록 표시 및 선택
if not uploaded:
    print("❌ 업로드된 파일이 없습니다.")
else:
    file_list = list(uploaded.keys())
    print("\n📂 업로드된 파일 목록:")
    for i, name in enumerate(file_list):
        print(f" [{i}] {name}")

    # 여러 개 업로드했을 경우 선택 (하나면 바로 진행)
    if len(file_list) > 1:
        idx = int(input("\n👉 분석할 파일의 번호를 입력하세요: "))
    else:
        idx = 0

    target_file = file_list[idx]

    # 3. 데이터 로드 및 통계 확인
    try:
        # 업로드된 바이너리 데이터를 판다스 데이터프레임으로 읽기
        df = pd.read_csv(io.BytesIO(uploaded[target_file]))

        if 'Label' in df.columns:
            # 쾌적(1)과 불쾌적(0) 개수 집계
            counts = df['Label'].value_counts().sort_index()

            pleasant = counts.get(1, 0)
            unpleasant = counts.get(0, 0)
            total = len(df)

            # 결과 출력
            print("\n" + "="*40)
            print(f"📊 [{target_file}] 라벨링 분석 결과")
            print("-" * 40)
            print(f"✨ 쾌적 (1)   : {pleasant:>6}개 ({pleasant/total*100:2.1f}%)")
            print(f"☁️  불쾌적 (0)  : {unpleasant:>6}개 ({unpleasant/total*100:2.1f}%)")
            print("-" * 40)
            print(f"📈 전체 데이터 : {total:>6}개")

            # 데이터 기간 확인 (Date_Key 컬럼이 있는 경우)
            if 'Date_Key' in df.columns:
                print(f"📅 데이터 기간 : {df['Date_Key'].iloc[0]} ~ {df['Date_Key'].iloc[-1]}")
            print("="*40)
        else:
            print(f"⚠️ 오류: '{target_file}' 파일에 'Label' 컬럼이 없습니다.")

    except Exception as e:
        print(f"❌ 분석 중 오류가 발생했습니다: {e}")

분석할 CSV 파일을 업로드해주세요.


Saving total_labeled_2013_2025 (7).csv to total_labeled_2013_2025 (7).csv

📂 업로드된 파일 목록:
 [0] total_labeled_2013_2025 (7).csv

📊 [total_labeled_2013_2025 (7).csv] 라벨링 분석 결과
----------------------------------------
✨ 쾌적 (1)   :    407개 (34.4%)
☁️  불쾌적 (0)  :    776개 (65.6%)
----------------------------------------
📈 전체 데이터 :   1183개
📅 데이터 기간 : 2013년09월_1일 ~ 2025년11월_30일
